In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import classification_report, confusion_matrix,accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
import os
import joblib


In [ ]:
print("Locating dataset...")
file_path = None
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        if filename == 'DNN-EdgeIIoT-dataset.csv':
            file_path = os.path.join(dirname, filename)
            break

if not file_path:
    raise FileNotFoundError("Could not find 'DNN-EdgeIIoT-dataset.csv'. Check your Kaggle inputs.")

print(f"Found dataset at: {file_path}")
print("Loading dataset (this may take a moment)...")
df = pd.read_csv(file_path, low_memory=False)
df.head()

In [ ]:
print("Dropping unnecessary flow features...")
drop_columns = [
    "frame.time", "ip.src_host", "ip.dst_host", "arp.src.proto_ipv4", 
    "arp.dst.proto_ipv4", "http.file_data", "http.request.full_uri", 
    "icmp.transmit_timestamp", "http.request.uri.query", "tcp.options", 
    "tcp.payload", "tcp.srcport", "tcp.dstport", "udp.port", "mqtt.msg"
]

df.drop(columns=[col for col in drop_columns if col in df.columns], inplace=True)

print("Dropping missing values and duplicates...")
df.dropna(axis=0, how='any', inplace=True)
df.drop_duplicates(subset=None, keep="first", inplace=True)

In [ ]:
df.head(5)

In [ ]:
print("Mapping labels for 2-class and 15-class tasks...")

# 15-Class
y_15 = df['Attack_type'].astype(str).values
le_15 = LabelEncoder()
y_15_encoded = le_15.fit_transform(y_15)

# 2-Class (Attack_label is 0 for Normal, 1 for Attack)
y_2 = df['Attack_label'].values

In [ ]:
X_df = df.drop(columns=['Attack_label', 'Attack_type'])

In [ ]:
print("Encoding categorical features...")
categorical_cols = ['http.request.method', 'http.referer', 'http.request.version', 
                    'dns.qry.name.len', 'mqtt.conack.flags', 'mqtt.protoname', 'mqtt.topic']

X_df = pd.get_dummies(X_df, columns=[col for col in categorical_cols if col in X_df.columns])

print("Finalizing feature matrix...")
X = X_df.apply(pd.to_numeric, errors='coerce').fillna(0).values


In [ ]:
print("Splitting data...")
X_train_2, X_test_2, y_train_2, y_test_2 = train_test_split(X, y_2, test_size=0.2, random_state=42, stratify=y_2)
X_train_15, X_test_15, y_train_15, y_test_15 = train_test_split(X, y_15_encoded, test_size=0.2, random_state=42, stratify=y_15_encoded)

In [ ]:
def evaluate_rf_model(X_train, y_train, X_test, y_test, class_names, task_name, filename):
    print(f"\n{'='*40}")
    print(f" TRAINING & EVALUATING: {task_name}")
    print(f"{'='*40}")
    
    # Force class names to string to avoid len() errors on floats/NaNs
    safe_class_names = [str(c) for c in class_names]
    
    # Initialize and Train Random Forest
    rf_model = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1, verbose=1)
    rf_model.fit(X_train, y_train)
    
    # Predict
    print("\nGenerating predictions...")
    y_pred = rf_model.predict(X_test)
        
    # Accuracy & Report
    acc = accuracy_score(y_test, y_pred)
    print(f"\nOverall Accuracy: {acc * 100:.4f}%\n")
    print("Classification Report:")
    print(classification_report(y_test, y_pred, target_names=safe_class_names, zero_division=0))
    
    # Plot Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', xticklabels=safe_class_names, yticklabels=safe_class_names)
    plt.title(f'Random Forest Confusion Matrix: {task_name}')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
    
    # Save Model
    joblib.dump(rf_model, filename)
    print(f"Model successfully saved to {filename}")

In [ ]:
evaluate_rf_model(X_train_15, y_train_15, X_test_15, y_test_15, le_15.classes_, '15-Class', 'rf_model_15_class.pkl')

In [ ]:
evaluate_rf_model(X_train_2, y_train_2, X_test_2, y_test_2, ['Normal', 'Attack'], '2-Class', 'rf_model_2_class.pkl')

In [ ]:
def evaluate_if_model(X_train, y_train, X_test, y_test, class_names, task_name, filename):
    print(f"\n{'='*40}")
    print(f" TRAINING & EVALUATING: {task_name}")
    print(f"{'='*40}")

    safe_class_names = [str(c) for c in class_names]

    # Обучаем IF только на нормальных объектах
    X_train_normal = X_train[y_train == 0]

    print(f"Training Isolation Forest only on normal traffic: {X_train_normal.shape[0]} samples")

    if_model = IsolationForest(
        n_estimators=100,
        contamination='auto',
        random_state=42,
        n_jobs=-1,
        verbose=1
    )

    if_model.fit(X_train_normal)

    print("\nGenerating predictions...")
    y_pred_raw = if_model.predict(X_test)

    # IsolationForest:
    #  1  -> normal
    # -1  -> anomaly
    # Приводим к твоей разметке:
    # 0 -> Normal
    # 1 -> Attack
    y_pred = np.where(y_pred_raw == 1, 0, 1)

    # Accuracy & Report
    acc = accuracy_score(y_test, y_pred)
    print(f"\nOverall Accuracy: {acc * 100:.4f}%\n")

    print("Classification Report:")
    print(classification_report(y_test, y_pred, target_names=safe_class_names, zero_division=0))

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Greens',
        xticklabels=safe_class_names,
        yticklabels=safe_class_names
    )
    plt.title(f'Isolation Forest Confusion Matrix: {task_name}')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.show()

    # Save model
    joblib.dump(if_model, filename)
    print(f"Model successfully saved to {filename}")

In [ ]:
evaluate_if_model(X_train_2, y_train_2, X_test_2, y_test_2, ['Normal', 'Attack'], 'IF', 'if_model.pkl')